# Milestone 3

This milestone is designed to familiarize us with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. We will then run side-by-side A/B tests to compare a model's zero-shot inference with and without this context. Finally, we will explore the physical limits of context windows (tuning the correct number of chunks to retrieve) and demonstrate the catastrophic dangers of feeding an LLM incorrect data.

### Milestone 3 setup

In [2]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('../data/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3450.67it/s]


Knowledge base successfully created


In [3]:
###              Zero-shot classifier for Q1, Q2, Q6


zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2288.91it/s]


### Question 1:

Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [7]:
result = zs(prompt_150, candidate_labels=labels_150)

score_map = dict(zip(result["labels"], result["scores"]))

ground_truth_score = score_map[ans_150]

print(round(ground_truth_score, 3))

0.384


### Question 2:

Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [11]:
# embed the prompt and query FAISS for top-10; report the 1-based rank of KB document at index 150
query_emb = model.encode([prompt_150], show_progress_bar=False)
distances, indices = index.search(query_emb, 10)
indices = indices[0]  # top-10 indices
target_idx = 150
try:
    rank = int(np.where(indices == target_idx)[0][0]) + 1
except IndexError:
    rank = None  # not in top-10
print(rank)

10


In [14]:
###      Code to use a Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6600.15it/s]


### Question 3:

Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [15]:
# Use existing ce_scores, indices, and target_idx to find cross-encoder rank (1-based)
order = np.argsort(-ce_scores)            # descending by score
sorted_indices = indices[order]
try:
    ce_rank = int(np.where(sorted_indices == target_idx)[0][0]) + 1
except IndexError:
    ce_rank = None
print(ce_rank)

1


### Question 4:

Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [16]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

query_emb_42 = model.encode([prompt_42], show_progress_bar=False)
_, indices_42 = index.search(query_emb_42, 5)
docs_5 = [kb[i] for i in indices_42[0]]

context = " ".join(docs_5)
text = f"Context: {context} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
num_tokens = len(tokenizer(text, truncation=False)["input_ids"])

print(num_tokens)

216


### Question 5:

Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [17]:
true_document = kb[150]
rag_text = f"Context: {true_document} Question: {prompt_150}"

rag_result = zs(rag_text, candidate_labels=labels_150)
rag_score_map = dict(zip(rag_result["labels"], rag_result["scores"]))

rag_ground_truth_score = rag_score_map[ans_150]
print(round(rag_ground_truth_score, 3))

0.989


### Question 6:

What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [18]:
adv_doc = kb[999]
adv_rag_text = f"Context: {adv_doc} Question: {prompt_150}"
adv_result = zs(adv_rag_text, candidate_labels=labels_150)
adv_score_map = dict(zip(adv_result["labels"], adv_result["scores"]))
adv_ground_truth_score = adv_score_map[ans_150]
print(round(adv_ground_truth_score, 3))

0.529


### Question 7:

In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [19]:
hit_count = 0

for idx in range(100):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_answer = str(row[row['answer']])
    
    # Embed the prompt and retrieve top-5 documents
    query_emb = model.encode([prompt], show_progress_bar=False)
    _, indices_top5 = index.search(query_emb, 5)
    
    # Check if correct answer is in any of the top-5 documents
    for doc_idx in indices_top5[0]:
        if correct_answer in kb[doc_idx]:
            hit_count += 1
            break

hit_rate = (hit_count / 100) * 100
print(round(hit_rate, 1))

73.0


### Question 8:

Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [20]:
map3_scores = []

for i in range(20):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    true_letter = row['answer']
    # Retrieve
    query_emb = model.encode([prompt], show_progress_bar=False)
    _, top5_idx = index.search(query_emb, 5)
    top5_idx = top5_idx[0]
    docs_5 = [kb[j] for j in top5_idx]
    # Rerank
    pairs = [[prompt, doc] for doc in docs_5]
    ce_scores_local = cross_encoder.predict(pairs)
    best_local = int(np.argmax(ce_scores_local))
    best_doc = docs_5[best_local]
    # Augment
    rag_text_local = f"Context: {best_doc} Question: {prompt}"
    # Predict
    candidate_labels = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    zs_res = zs(rag_text_local, candidate_labels=candidate_labels)
    scores = np.array(zs_res["scores"])
    order = np.argsort(-scores)  # descending
    top3_texts = [zs_res["labels"][idx] for idx in order[:3]]
    # Map texts back to letters
    text_to_letter = {str(row['A']): 'A', str(row['B']): 'B', str(row['C']): 'C', str(row['D']): 'D', str(row['E']): 'E'}
    top3_letters = [text_to_letter.get(t) for t in top3_texts]
    # MAP@3 for this row (single relevant item)
    if true_letter in top3_letters:
        pos = top3_letters.index(true_letter)
        ap = 1.0 / (pos + 1)
    else:
        ap = 0.0
    map3_scores.append(ap)

avg_map3 = float(np.mean(map3_scores))
print(round(avg_map3, 3))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


0.975
